# Porto Taxi ETA Exploration

Notebook nay mo dau cho bai toan du doan ETA tu GPS trajectory theo thoi gian thuc. Muc tieu cua chung ta la doc du lieu goc `train.csv`, hieu schema, va bien moi chuyen di thanh nhung dac trung co the dung de du doan thoi gian con lai den dich.

In [7]:
import ast
import math
from pathlib import Path

import pandas as pd

DATA_PATH = Path("../train.csv")
SAMPLE_ROWS = 100000

df = pd.read_csv(DATA_PATH, nrows=SAMPLE_ROWS)
print(f"Loaded {len(df):,} rows from {DATA_PATH.resolve()}")
df.head(3)

Loaded 100,000 rows from /home/duy/ml_project/train.csv


,TRIP_ID,CALL_TYPE,ORIGIN_CALL,ORIGIN_STAND,TAXI_ID,TIMESTAMP,DAY_TYPE,MISSING_DATA,POLYLINE
0,1372636858620000589,C,NaN,NaN,20000589,1372636858,A,False,"[[-8.618643,41.141412],[-8.618499,41.141376],[..."
1,1372637303620000596,B,NaN,7.0,20000596,1372637303,A,False,"[[-8.639847,41.159826],[-8.640351,41.159871],[..."
2,1372636951620000320,C,NaN,NaN,20000320,1372636951,A,False,"[[-8.612964,41.140359],[-8.613378,41.14035],[-..."


## Kiem tra schema ban dau

Sau khi doc du lieu, viec dau tien la xac nhan kich thuoc, ten cot, va kieu du lieu. Buoc nay giup minh thay ngay cot nao la metadata cua chuyen di, cot nao chua thong tin GPS trajectory, va cot nao co the dung de tao feature cho ETA.

In [8]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nDtypes:")
display(df.dtypes)
display(df.head(3))

Shape: (100000, 9)

Columns:
['TRIP_ID', 'CALL_TYPE', 'ORIGIN_CALL', 'ORIGIN_STAND', 'TAXI_ID', 'TIMESTAMP', 'DAY_TYPE', 'MISSING_DATA', 'POLYLINE']

Dtypes:


TRIP_ID           int64
CALL_TYPE           str
ORIGIN_CALL     float64
ORIGIN_STAND    float64
TAXI_ID           int64
TIMESTAMP         int64
DAY_TYPE            str
MISSING_DATA       bool
POLYLINE            str
dtype: object

,TRIP_ID,CALL_TYPE,ORIGIN_CALL,ORIGIN_STAND,TAXI_ID,TIMESTAMP,DAY_TYPE,MISSING_DATA,POLYLINE
0,1372636858620000589,C,NaN,NaN,20000589,1372636858,A,False,"[[-8.618643,41.141412],[-8.618499,41.141376],[..."
1,1372637303620000596,B,NaN,7.0,20000596,1372637303,A,False,"[[-8.639847,41.159826],[-8.640351,41.159871],[..."
2,1372636951620000320,C,NaN,NaN,20000320,1372636951,A,False,"[[-8.612964,41.140359],[-8.613378,41.14035],[-..."


## Kiem tra chat luong du lieu

Truoc khi parse trajectory, can kiem tra nhanh do day du cua du lieu. Trong bo Porto Taxi, `MISSING_DATA` rat quan trong vi no bao hieu mot chuyen di bi thieu GPS point. Neu dung de train ETA, nhung dong nay thuong nen loc ra o giai doan tien xu ly.

In [9]:
missing_per_column = df.isna().sum().sort_values(ascending=False)
missing_data_counts = df["MISSING_DATA"].value_counts(dropna=False)
empty_polyline_count = (df["POLYLINE"] == "[]").sum()

display(missing_per_column)
display(missing_data_counts)
print(f"Empty POLYLINE rows: {empty_polyline_count:,}")

ORIGIN_CALL     79038
ORIGIN_STAND    50689
TRIP_ID             0
CALL_TYPE           0
TAXI_ID             0
TIMESTAMP           0
DAY_TYPE            0
MISSING_DATA        0
POLYLINE            0
dtype: int64

MISSING_DATA
False    100000
Name: count, dtype: int64

Empty POLYLINE rows: 367


## Parse `POLYLINE` va tao nhan ETA co ban

Moi dong trong `POLYLINE` la mot danh sach cac cap toa do `[longitude, latitude]`, lay mau moi 15 giay. Vi vay neu mot trajectory co `n` diem thi tong thoi gian chuyen di xap xi bang `(n - 1) * 15` giay. Day la cach don gian nhat de tao nhan ETA tu du lieu goc.

In [10]:
def parse_polyline(polyline_text: str):
    points = ast.literal_eval(polyline_text)
    return points if isinstance(points, list) else []


sample = df.loc[:, [
    "TRIP_ID",
    "TAXI_ID",
    "TIMESTAMP",
    "MISSING_DATA",
    "POLYLINE",
]].copy()
sample["parsed_polyline"] = sample["POLYLINE"].apply(parse_polyline)
sample["num_points"] = sample["parsed_polyline"].apply(len)
sample["trip_duration_seconds"] = (sample["num_points"] - 1).clip(lower=0) * 15

preview_columns = [
    "TRIP_ID",
    "TIMESTAMP",
    "MISSING_DATA",
    "num_points",
    "trip_duration_seconds",
]

display(sample[preview_columns].head(10))
print("\nTrip duration summary (seconds):")
display(sample["trip_duration_seconds"].describe())

,TRIP_ID,TIMESTAMP,MISSING_DATA,num_points,trip_duration_seconds
0,1372636858620000589,1372636858,False,23,330
1,1372637303620000596,1372637303,False,19,270
2,1372636951620000320,1372636951,False,65,960
3,1372636854620000520,1372636854,False,43,630
4,1372637091620000337,1372637091,False,29,420
5,1372636965620000231,1372636965,False,26,375
6,1372637210620000456,1372637210,False,36,525
7,1372637299620000011,1372637299,False,34,495
8,1372637274620000403,1372637274,False,38,555
9,1372637905620000320,1372637905,False,19,270



Trip duration summary (seconds):


count    100000.000000
mean        701.306700
std         659.930272
min           0.000000
25%         405.000000
50%         600.000000
75%         840.000000
max       37725.000000
Name: trip_duration_seconds, dtype: float64

## Loc du lieu sach va tao feature thoi gian

De phuc vu bai toan ETA, ta nen bat dau tu nhung chuyen di co trajectory day du. O buoc nay, minh loai bo cac dong co `MISSING_DATA=True` hoac trajectory qua ngan, sau do chuyen `TIMESTAMP` sang datetime de tao mot vai feature don gian nhu gio bat dau, thu trong tuan, va weekend. Notebook nay chi nen explore tren mot tap con du lieu; neu xu ly full dataset thi nen chay theo batch.

In [11]:
clean_df = sample.copy()
clean_df["missing_flag"] = clean_df["MISSING_DATA"].astype(str).str.lower() == "true"
clean_df = clean_df[(~clean_df["missing_flag"]) & (clean_df["num_points"] > 1)].copy()

clean_df["start_datetime"] = pd.to_datetime(clean_df["TIMESTAMP"], unit="s")
clean_df["start_hour"] = clean_df["start_datetime"].dt.hour
clean_df["day_of_week"] = clean_df["start_datetime"].dt.dayofweek
clean_df["is_weekend"] = clean_df["day_of_week"].isin([5, 6])

print(f"Clean rows kept: {len(clean_df):,} / {len(sample):,}")
display(
    clean_df[[
        "TRIP_ID",
        "start_datetime",
        "start_hour",
        "day_of_week",
        "is_weekend",
        "num_points",
        "trip_duration_seconds",
    ]].head(10)
)

Clean rows kept: 98,445 / 100,000


,TRIP_ID,start_datetime,start_hour,day_of_week,is_weekend,num_points,trip_duration_seconds
0,1372636858620000589,2013-07-01 00:00:58,0,0,False,23,330
1,1372637303620000596,2013-07-01 00:08:23,0,0,False,19,270
2,1372636951620000320,2013-07-01 00:02:31,0,0,False,65,960
3,1372636854620000520,2013-07-01 00:00:54,0,0,False,43,630
4,1372637091620000337,2013-07-01 00:04:51,0,0,False,29,420
5,1372636965620000231,2013-07-01 00:02:45,0,0,False,26,375
6,1372637210620000456,2013-07-01 00:06:50,0,0,False,36,525
7,1372637299620000011,2013-07-01 00:08:19,0,0,False,34,495
8,1372637274620000403,2013-07-01 00:07:54,0,0,False,38,555
9,1372637905620000320,2013-07-01 00:18:25,0,0,False,19,270


## Bien 1 trip thanh nhieu sample ETA

Day la buoc chuyen tu du lieu raw sang bai toan ETA thoi gian thuc. Thay vi dua ca chuyen di hoan chinh vao model, ta cat moi chuyen di thanh nhieu `prefix trajectory`. Moi prefix dong vai tro la trang thai hien tai cua xe, con nhan `remaining_eta_seconds` chinh la thoi gian con lai den dich.

In [12]:
def haversine_distance_km(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(math.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    c = 2 * math.asin(math.sqrt(a))
    earth_radius_km = 6371.0
    return earth_radius_km * c


def build_prefix_samples(points, trip_id, min_prefix_points=3, step=5):
    rows = []
    total_points = len(points)
    start_lon, start_lat = points[0]
    dest_lon, dest_lat = points[-1]

    for prefix_points in range(min_prefix_points, total_points, step):
        observed = points[:prefix_points]
        remaining_segments = total_points - prefix_points
        elapsed_seconds = (prefix_points - 1) * 15
        current_lon, current_lat = observed[-1]
        distance_to_destination = haversine_distance_km(current_lon, current_lat, dest_lon, dest_lat)
        distance_from_start = haversine_distance_km(start_lon, start_lat, current_lon, current_lat)
        total_trip_distance_proxy = distance_from_start + distance_to_destination
        progress_ratio = distance_from_start / total_trip_distance_proxy if total_trip_distance_proxy > 0 else 0.0
        avg_speed_since_start_kmh = distance_from_start / (elapsed_seconds / 3600)

        prev_lon, prev_lat = observed[-2]
        recent_distance_km = haversine_distance_km(prev_lon, prev_lat, current_lon, current_lat)
        recent_speed_kmh = recent_distance_km / (15 / 3600)

        segment_speeds = []
        recent_points = observed[-4:]
        for i in range(1, len(recent_points)):
            lon_a, lat_a = recent_points[i - 1]
            lon_b, lat_b = recent_points[i]
            segment_distance_km = haversine_distance_km(lon_a, lat_a, lon_b, lat_b)
            segment_speed_kmh = segment_distance_km / (15 / 3600)
            segment_speeds.append(segment_speed_kmh)

        recent_speed_avg_kmh = sum(segment_speeds) / len(segment_speeds)

        rows.append(
            {
                "TRIP_ID": trip_id,
                "observed_points": prefix_points,
                "remaining_points": remaining_segments,
                "remaining_eta_seconds": remaining_segments * 15,
                "start_longitude": start_lon,
                "start_latitude": start_lat,
                "current_longitude": current_lon,
                "current_latitude": current_lat,
                "destination_longitude": dest_lon,
                "destination_latitude": dest_lat,
                "distance_to_destination": distance_to_destination,
                "distance_from_start": distance_from_start,
                "progress_ratio": progress_ratio,
                "avg_speed_since_start_kmh": avg_speed_since_start_kmh,
                "recent_speed_kmh": recent_speed_kmh,
                "recent_speed_avg_kmh": recent_speed_avg_kmh,
            }
        )

    return rows


example_trip = clean_df.iloc[0]
example_points = example_trip["parsed_polyline"]
prefix_samples = build_prefix_samples(example_points, example_trip["TRIP_ID"])
prefix_df = pd.DataFrame(prefix_samples)

print(f"Example trip id: {example_trip['TRIP_ID']}")
print(f"Total GPS points in trip: {len(example_points)}")
display(prefix_df.head(10))

Example trip id: 1372636858620000589
Total GPS points in trip: 23


,TRIP_ID,observed_points,remaining_points,remaining_eta_seconds,start_longitude,start_latitude,current_longitude,current_latitude,destination_longitude,destination_latitude,distance_to_destination,distance_from_start,progress_ratio,avg_speed_since_start_kmh,recent_speed_kmh,recent_speed_avg_kmh
0,1372636858620000589,3,20,300,-8.618643,41.141412,-8.620326,41.142510,-8.630838,41.154489,1.596541,0.186463,0.104578,22.375556,47.581896,25.315617
1,1372636858620000589,8,15,225,-8.618643,41.141412,-8.630226,41.145210,-8.630838,41.154489,1.033049,1.057881,0.505938,36.270200,58.946877,42.966445
2,1372636858620000589,13,10,150,-8.618643,41.141412,-8.629128,41.151240,-8.630838,41.154489,0.388607,1.401802,0.782951,28.036039,0.806232,32.172979
3,1372636858620000589,18,5,75,-8.618643,41.141412,-8.632323,41.153022,-8.630838,41.154489,0.205102,1.725886,0.893784,24.365443,31.204171,25.769688


## Tao prefix dataset tu nhieu trip

Sau khi da hieu logic tren 1 chuyen di, buoc tiep theo la ap dung no cho nhieu trip de tao ra dataset train that su. De tranh lam treo notebook, o day minh chi sinh prefix tren mot so luong trip gioi han. Khi can xu ly full dataset, minh se chuyen sang cach doc theo chunk va ghi ket qua ra file trung gian.

In [13]:
MAX_TRIPS_FOR_PREFIX = 100000
MIN_PREFIX_POINTS = 3
PREFIX_STEP = 5

prefix_rows = []
selected_trips = clean_df.head(MAX_TRIPS_FOR_PREFIX)

for trip in selected_trips.itertuples(index=False):
    trip_prefixes = build_prefix_samples(
        points=trip.parsed_polyline,
        trip_id=trip.TRIP_ID,
        min_prefix_points=MIN_PREFIX_POINTS,
        step=PREFIX_STEP,
    )

    for row in trip_prefixes:
        row["TAXI_ID"] = trip.TAXI_ID
        row["start_hour"] = trip.start_hour
        row["day_of_week"] = trip.day_of_week
        row["is_weekend"] = trip.is_weekend
        row["trip_duration_seconds"] = trip.trip_duration_seconds
        prefix_rows.append(row)

prefix_dataset = pd.DataFrame(prefix_rows)

print(f"Trips used: {len(selected_trips):,}")
print(f"Prefix samples created: {len(prefix_dataset):,}")
display(prefix_dataset.head(10))

Trips used: 98,445
Prefix samples created: 934,906


,TRIP_ID,observed_points,remaining_points,remaining_eta_seconds,start_longitude,start_latitude,current_longitude,current_latitude,destination_longitude,destination_latitude,...,distance_from_start,progress_ratio,avg_speed_since_start_kmh,recent_speed_kmh,recent_speed_avg_kmh,TAXI_ID,start_hour,day_of_week,is_weekend,trip_duration_seconds
0,1372636858620000589,3,20,300,-8.618643,41.141412,-8.620326,41.142510,-8.630838,41.154489,...,0.186463,0.104578,22.375556,47.581896,25.315617,20000589,0,0,False,330
1,1372636858620000589,8,15,225,-8.618643,41.141412,-8.630226,41.145210,-8.630838,41.154489,...,1.057881,0.505938,36.270200,58.946877,42.966445,20000589,0,0,False,330
2,1372636858620000589,13,10,150,-8.618643,41.141412,-8.629128,41.151240,-8.630838,41.154489,...,1.401802,0.782951,28.036039,0.806232,32.172979,20000589,0,0,False,330
3,1372636858620000589,18,5,75,-8.618643,41.141412,-8.632323,41.153022,-8.630838,41.154489,...,1.725886,0.893784,24.365443,31.204171,25.769688,20000589,0,0,False,330
4,1372637303620000596,3,16,240,-8.639847,41.159826,-8.642196,41.160114,-8.665740,41.170671,...,0.199239,0.079913,23.908689,37.632331,23.914791,20000596,0,0,False,270
5,1372637303620000596,8,11,165,-8.639847,41.159826,-8.656434,41.162580,-8.665740,41.170671,...,1.421936,0.544388,48.752083,67.252865,65.337089,20000596,0,0,False,270
6,1372637303620000596,13,6,90,-8.639847,41.159826,-8.670852,41.165136,-8.665740,41.170671,...,2.661823,0.780268,53.236467,35.723373,53.467636,20000596,0,0,False,270
7,1372637303620000596,18,1,15,-8.639847,41.159826,-8.665767,41.170635,-8.665740,41.170671,...,2.480394,0.998150,35.017328,24.294560,35.039908,20000596,0,0,False,270
8,1372636951620000320,3,62,930,-8.612964,41.140359,-8.614215,41.140278,-8.615970,41.140530,...,0.105146,0.412729,12.617570,16.931268,12.627616,20000320,0,0,False,960
9,1372636951620000320,8,57,855,-8.612964,41.140359,-8.620623,41.142789,-8.615970,41.140530,...,0.695954,0.600197,23.861291,56.737664,38.219668,20000320,0,0,False,960


## Mau xu ly full dataset theo batch

Neu muon di het `train.csv`, dung mo rong cell o tren trong notebook la rat nguy hiem vi RAM se tang manh khi parse `POLYLINE`. Cach an toan hon la doc file theo `chunksize`, parse tung chunk, tao prefix, roi ghi dan ket qua ra file `.parquet` hoac `.csv` trung gian.

In [14]:
CHUNK_SIZE = 20_000

print("Pseudo-pipeline for full preprocessing:")
print("1. Read train.csv with chunksize=20_000")
print("2. Filter valid trips inside each chunk")
print("3. Parse POLYLINE only for that chunk")
print("4. Build prefix rows and append to an output file")
print("5. After all chunks finish, read the output file for training")

Pseudo-pipeline for full preprocessing:
1. Read train.csv with chunksize=20_000
2. Filter valid trips inside each chunk
3. Parse POLYLINE only for that chunk
4. Build prefix rows and append to an output file
5. After all chunks finish, read the output file for training


## Xuat DataFrame san sang cho baseline model

Dataset vua tao da co dang tabular, moi dong la mot trang thai hien tai cua xe va nhan `remaining_eta_seconds`. Minh se chon mot tap feature co ban va tao ra `baseline_df` de buoc sau co the dua thang vao model hoi quy nhu Linear Regression, Random Forest, hoac XGBoost.

In [15]:
baseline_columns = [
    "TRIP_ID",
    "TAXI_ID",
    "observed_points",
    "remaining_points",
    "start_longitude",
    "start_latitude",
    "current_longitude",
    "current_latitude",
    "destination_longitude",
    "destination_latitude",
    "distance_to_destination",
    "distance_from_start",
    "progress_ratio",
    "avg_speed_since_start_kmh",
    "recent_speed_kmh",
    "recent_speed_avg_kmh",
    "start_hour",
    "day_of_week",
    "is_weekend",
    "trip_duration_seconds",
    "remaining_eta_seconds",
]

baseline_df = prefix_dataset[baseline_columns].copy()

print("Baseline dataset shape:", baseline_df.shape)
display(baseline_df.head(10))

Baseline dataset shape: (934906, 21)


,TRIP_ID,TAXI_ID,observed_points,remaining_points,start_longitude,start_latitude,current_longitude,current_latitude,destination_longitude,destination_latitude,...,distance_from_start,progress_ratio,avg_speed_since_start_kmh,recent_speed_kmh,recent_speed_avg_kmh,start_hour,day_of_week,is_weekend,trip_duration_seconds,remaining_eta_seconds
0,1372636858620000589,20000589,3,20,-8.618643,41.141412,-8.620326,41.142510,-8.630838,41.154489,...,0.186463,0.104578,22.375556,47.581896,25.315617,0,0,False,330,300
1,1372636858620000589,20000589,8,15,-8.618643,41.141412,-8.630226,41.145210,-8.630838,41.154489,...,1.057881,0.505938,36.270200,58.946877,42.966445,0,0,False,330,225
2,1372636858620000589,20000589,13,10,-8.618643,41.141412,-8.629128,41.151240,-8.630838,41.154489,...,1.401802,0.782951,28.036039,0.806232,32.172979,0,0,False,330,150
3,1372636858620000589,20000589,18,5,-8.618643,41.141412,-8.632323,41.153022,-8.630838,41.154489,...,1.725886,0.893784,24.365443,31.204171,25.769688,0,0,False,330,75
4,1372637303620000596,20000596,3,16,-8.639847,41.159826,-8.642196,41.160114,-8.665740,41.170671,...,0.199239,0.079913,23.908689,37.632331,23.914791,0,0,False,270,240
5,1372637303620000596,20000596,8,11,-8.639847,41.159826,-8.656434,41.162580,-8.665740,41.170671,...,1.421936,0.544388,48.752083,67.252865,65.337089,0,0,False,270,165
6,1372637303620000596,20000596,13,6,-8.639847,41.159826,-8.670852,41.165136,-8.665740,41.170671,...,2.661823,0.780268,53.236467,35.723373,53.467636,0,0,False,270,90
7,1372637303620000596,20000596,18,1,-8.639847,41.159826,-8.665767,41.170635,-8.665740,41.170671,...,2.480394,0.998150,35.017328,24.294560,35.039908,0,0,False,270,15
8,1372636951620000320,20000320,3,62,-8.612964,41.140359,-8.614215,41.140278,-8.615970,41.140530,...,0.105146,0.412729,12.617570,16.931268,12.627616,0,0,False,960,930
9,1372636951620000320,20000320,8,57,-8.612964,41.140359,-8.620623,41.142789,-8.615970,41.140530,...,0.695954,0.600197,23.861291,56.737664,38.219668,0,0,False,960,855


## Chia train validation theo `TRIP_ID`

Neu mot `TRIP_ID` xuat hien o ca train va validation thi model se thay mot phan hanh trinh cua cung mot chuyen di o ca hai ben. Dieu do gay leakage. Vi vay minh chia theo nhom `TRIP_ID` thay vi chia ngau nhien tung dong.

In [16]:
from sklearn.model_selection import GroupShuffleSplit

feature_columns = [
    "observed_points",
    "start_longitude",
    "start_latitude",
    "current_longitude",
    "current_latitude",
    "destination_longitude",
    "destination_latitude",
    "distance_to_destination",
    "distance_from_start",
    "progress_ratio",
    "avg_speed_since_start_kmh",
    "recent_speed_kmh",
    "recent_speed_avg_kmh",
    "start_hour",
    "day_of_week",
    "is_weekend",
]
target_column = "remaining_eta_seconds"

X = baseline_df[feature_columns].copy()
y = baseline_df[target_column].copy()
groups = baseline_df["TRIP_ID"].copy()

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, valid_idx = next(splitter.split(X, y, groups=groups))

X_train = X.iloc[train_idx]
X_valid = X.iloc[valid_idx]
y_train = y.iloc[train_idx]
y_valid = y.iloc[valid_idx]

print("Train shape:", X_train.shape)
print("Validation shape:", X_valid.shape)
print("Unique train trips:", groups.iloc[train_idx].nunique())
print("Unique validation trips:", groups.iloc[valid_idx].nunique())

Train shape: (747226, 16)
Validation shape: (187680, 16)
Unique train trips: 78268
Unique validation trips: 19568


## Train baseline `LinearRegression`

O day minh chu dong bo feature `trip_duration_seconds` ra khoi tap train. Ly do la cot nay duoc tinh tu toan bo trajectory cua chuyen di, trong khi o bai toan real-time thi tai thoi diem hien tai ta chua biet tong do dai cua ca chuyen. Giu cot nay se lam lo nhan ETA va cho ra ket qua dep gia tao.

In [17]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

valid_predictions = linear_model.predict(X_valid)

mae = mean_absolute_error(y_valid, valid_predictions)
rmse = mean_squared_error(y_valid, valid_predictions) ** 0.5

print(f"Validation MAE: {mae:.2f} seconds ({mae / 60:.2f} minutes)")
print(f"Validation RMSE: {rmse:.2f} seconds ({rmse / 60:.2f} minutes)")

comparison_df = X_valid.copy()
comparison_df["actual_eta_seconds"] = y_valid.values
comparison_df["predicted_eta_seconds"] = valid_predictions
comparison_df["absolute_error_seconds"] = (
    comparison_df["actual_eta_seconds"] - comparison_df["predicted_eta_seconds"]
).abs()

display(comparison_df.head(10))

Validation MAE: 405.70 seconds (6.76 minutes)
Validation RMSE: 1301.35 seconds (21.69 minutes)


,observed_points,start_longitude,start_latitude,current_longitude,current_latitude,destination_longitude,destination_latitude,distance_to_destination,distance_from_start,progress_ratio,avg_speed_since_start_kmh,recent_speed_kmh,recent_speed_avg_kmh,start_hour,day_of_week,is_weekend,actual_eta_seconds,predicted_eta_seconds,absolute_error_seconds
176,3,-8.613243,41.154444,-8.611965,41.154642,-8.613315,41.166891,1.366707,0.109239,0.074013,13.108728,29.621549,25.243586,0,0,False,225,647.693832,422.693832
177,8,-8.613243,41.154444,-8.611371,41.161581,-8.613315,41.166891,0.612460,0.808925,0.569110,27.734568,31.100578,38.698648,0,0,False,150,279.885557,129.885557
178,13,-8.613243,41.154444,-8.611470,41.166378,-8.613315,41.166891,0.164638,1.335275,0.890235,26.705507,40.413846,20.677150,0,0,False,75,61.348194,13.651806
200,3,-8.609967,41.140827,-8.609841,41.138883,-8.641998,41.131953,2.801136,0.216420,0.071720,25.970437,20.875967,28.403099,0,0,False,495,723.573689,228.573689
201,8,-8.609967,41.140827,-8.618004,41.132790,-8.641998,41.131953,2.011678,1.118778,0.357385,38.358119,74.294458,54.734207,0,0,False,420,480.986400,60.986400
202,13,-8.609967,41.140827,-8.624277,41.129829,-8.641998,41.131953,1.502861,1.712238,0.532562,34.244759,20.471621,36.000089,0,0,False,345,369.024062,24.024062
203,18,-8.609967,41.140827,-8.629209,41.127111,-8.641998,41.131953,1.198841,2.218786,0.649218,31.324042,18.473653,34.102044,0,0,False,270,292.530831,22.530831
204,23,-8.609967,41.140827,-8.634033,41.126229,-8.641998,41.131953,0.922032,2.587889,0.737307,28.231519,31.844399,34.903556,0,0,False,195,232.254441,37.254441
205,28,-8.609967,41.140827,-8.643195,41.129055,-8.641998,41.131953,0.337478,3.075268,0.901113,27.335713,47.811084,40.940495,0,0,False,120,90.994740,29.005260
206,33,-8.609967,41.140827,-8.642871,41.132313,-8.641998,41.131953,0.083356,2.913663,0.972187,21.852475,34.206879,30.638532,0,0,False,45,58.462814,13.462814


## Train baseline `XGBoost`

Sau khi co ket qua tu `LinearRegression`, minh co the thu `XGBoost` de xem model phi tuyen co hoc duoc quan he tot hon hay khong. Cell nay dung lai cung tap `X_train`, `X_valid`, `y_train`, `y_valid` va cung cach danh gia `MAE`, `RMSE` de viec so sanh duoc cong bang.

In [18]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.02,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)

xgb_model.fit(X_train, y_train)
xgb_valid_predictions = xgb_model.predict(X_valid)

xgb_mae = mean_absolute_error(y_valid, xgb_valid_predictions)
xgb_rmse = mean_squared_error(y_valid, xgb_valid_predictions) ** 0.5

print(f"XGBoost Validation MAE: {xgb_mae:.2f} seconds ({xgb_mae / 60:.2f} minutes)")
print(f"XGBoost Validation RMSE: {xgb_rmse:.2f} seconds ({xgb_rmse / 60:.2f} minutes)")

xgb_comparison_df = X_valid.copy()
xgb_comparison_df["actual_eta_seconds"] = y_valid.values
xgb_comparison_df["predicted_eta_seconds"] = xgb_valid_predictions
xgb_comparison_df["absolute_error_seconds"] = (
    xgb_comparison_df["actual_eta_seconds"] - xgb_comparison_df["predicted_eta_seconds"]
).abs()

display(xgb_comparison_df.head(10))

XGBoost Validation MAE: 372.49 seconds (6.21 minutes)
XGBoost Validation RMSE: 1412.62 seconds (23.54 minutes)


,observed_points,start_longitude,start_latitude,current_longitude,current_latitude,destination_longitude,destination_latitude,distance_to_destination,distance_from_start,progress_ratio,avg_speed_since_start_kmh,recent_speed_kmh,recent_speed_avg_kmh,start_hour,day_of_week,is_weekend,actual_eta_seconds,predicted_eta_seconds,absolute_error_seconds
176,3,-8.613243,41.154444,-8.611965,41.154642,-8.613315,41.166891,1.366707,0.109239,0.074013,13.108728,29.621549,25.243586,0,0,False,225,348.727142,123.727142
177,8,-8.613243,41.154444,-8.611371,41.161581,-8.613315,41.166891,0.612460,0.808925,0.569110,27.734568,31.100578,38.698648,0,0,False,150,243.761551,93.761551
178,13,-8.613243,41.154444,-8.611470,41.166378,-8.613315,41.166891,0.164638,1.335275,0.890235,26.705507,40.413846,20.677150,0,0,False,75,110.997452,35.997452
200,3,-8.609967,41.140827,-8.609841,41.138883,-8.641998,41.131953,2.801136,0.216420,0.071720,25.970437,20.875967,28.403099,0,0,False,495,516.217712,21.217712
201,8,-8.609967,41.140827,-8.618004,41.132790,-8.641998,41.131953,2.011678,1.118778,0.357385,38.358119,74.294458,54.734207,0,0,False,420,435.155090,15.155090
202,13,-8.609967,41.140827,-8.624277,41.129829,-8.641998,41.131953,1.502861,1.712238,0.532562,34.244759,20.471621,36.000089,0,0,False,345,341.309570,3.690430
203,18,-8.609967,41.140827,-8.629209,41.127111,-8.641998,41.131953,1.198841,2.218786,0.649218,31.324042,18.473653,34.102044,0,0,False,270,281.475739,11.475739
204,23,-8.609967,41.140827,-8.634033,41.126229,-8.641998,41.131953,0.922032,2.587889,0.737307,28.231519,31.844399,34.903556,0,0,False,195,227.553528,32.553528
205,28,-8.609967,41.140827,-8.643195,41.129055,-8.641998,41.131953,0.337478,3.075268,0.901113,27.335713,47.811084,40.940495,0,0,False,120,130.288849,10.288849
206,33,-8.609967,41.140827,-8.642871,41.132313,-8.641998,41.131953,0.083356,2.913663,0.972187,21.852475,34.206879,30.638532,0,0,False,45,67.528557,22.528557


## Train `XGBoost` voi `log1p(target)`

Nhan ETA co duoi phan phoi rat nang, nhat la o cac trip dai. Mot cach don gian de giam anh huong cua outlier la train tren `log1p(remaining_eta_seconds)`, sau do doi nguoc ve giay bang `expm1` khi predict.

In [19]:
import numpy as np

y_train_log = np.log1p(y_train)

xgb_log_model = XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)

xgb_log_model.fit(X_train, y_train_log)
xgb_log_valid_predictions = np.expm1(xgb_log_model.predict(X_valid))
xgb_log_valid_predictions = np.clip(xgb_log_valid_predictions, a_min=0, a_max=None)

xgb_log_mae = mean_absolute_error(y_valid, xgb_log_valid_predictions)
xgb_log_rmse = mean_squared_error(y_valid, xgb_log_valid_predictions) ** 0.5

print(f"XGBoost Log-Target Validation MAE: {xgb_log_mae:.2f} seconds ({xgb_log_mae / 60:.2f} minutes)")
print(f"XGBoost Log-Target Validation RMSE: {xgb_log_rmse:.2f} seconds ({xgb_log_rmse / 60:.2f} minutes)")

xgb_log_comparison_df = X_valid.copy()
xgb_log_comparison_df["actual_eta_seconds"] = y_valid.values
xgb_log_comparison_df["predicted_eta_seconds"] = xgb_log_valid_predictions
xgb_log_comparison_df["absolute_error_seconds"] = (
    xgb_log_comparison_df["actual_eta_seconds"] - xgb_log_comparison_df["predicted_eta_seconds"]
).abs()

display(xgb_log_comparison_df.head(10))

XGBoost Log-Target Validation MAE: 332.50 seconds (5.54 minutes)
XGBoost Log-Target Validation RMSE: 1478.87 seconds (24.65 minutes)


,observed_points,start_longitude,start_latitude,current_longitude,current_latitude,destination_longitude,destination_latitude,distance_to_destination,distance_from_start,progress_ratio,avg_speed_since_start_kmh,recent_speed_kmh,recent_speed_avg_kmh,start_hour,day_of_week,is_weekend,actual_eta_seconds,predicted_eta_seconds,absolute_error_seconds
176,3,-8.613243,41.154444,-8.611965,41.154642,-8.613315,41.166891,1.366707,0.109239,0.074013,13.108728,29.621549,25.243586,0,0,False,225,298.220551,73.220551
177,8,-8.613243,41.154444,-8.611371,41.161581,-8.613315,41.166891,0.612460,0.808925,0.569110,27.734568,31.100578,38.698648,0,0,False,150,150.701965,0.701965
178,13,-8.613243,41.154444,-8.611470,41.166378,-8.613315,41.166891,0.164638,1.335275,0.890235,26.705507,40.413846,20.677150,0,0,False,75,71.151909,3.848091
200,3,-8.609967,41.140827,-8.609841,41.138883,-8.641998,41.131953,2.801136,0.216420,0.071720,25.970437,20.875967,28.403099,0,0,False,495,499.112579,4.112579
201,8,-8.609967,41.140827,-8.618004,41.132790,-8.641998,41.131953,2.011678,1.118778,0.357385,38.358119,74.294458,54.734207,0,0,False,420,329.798950,90.201050
202,13,-8.609967,41.140827,-8.624277,41.129829,-8.641998,41.131953,1.502861,1.712238,0.532562,34.244759,20.471621,36.000089,0,0,False,345,302.993896,42.006104
203,18,-8.609967,41.140827,-8.629209,41.127111,-8.641998,41.131953,1.198841,2.218786,0.649218,31.324042,18.473653,34.102044,0,0,False,270,239.143219,30.856781
204,23,-8.609967,41.140827,-8.634033,41.126229,-8.641998,41.131953,0.922032,2.587889,0.737307,28.231519,31.844399,34.903556,0,0,False,195,170.550781,24.449219
205,28,-8.609967,41.140827,-8.643195,41.129055,-8.641998,41.131953,0.337478,3.075268,0.901113,27.335713,47.811084,40.940495,0,0,False,120,88.121216,31.878784
206,33,-8.609967,41.140827,-8.642871,41.132313,-8.641998,41.131953,0.083356,2.913663,0.972187,21.852475,34.206879,30.638532,0,0,False,45,45.775108,0.775108


## Xem cac dong du doan te nhat

Cell nay sap xep theo `absolute_error_seconds` giam dan de tim cac truong hop du doan te nhat. De so sanh nhanh, minh chi xuat 3 cot cuoi la `actual_eta_seconds`, `predicted_eta_seconds`, va `absolute_error_seconds`.

In [20]:
worst_rows = xgb_comparison_df.sort_values("absolute_error_seconds", ascending=False).head(20)
display(worst_rows[["actual_eta_seconds", "predicted_eta_seconds", "absolute_error_seconds"]])

,actual_eta_seconds,predicted_eta_seconds,absolute_error_seconds
116184,37695,8957.878906,28737.121094
116193,37020,8317.731445,28702.268555
116185,37620,8951.782227,28668.217773
116195,36870,8234.305664,28635.694336
116194,36945,8318.088867,28626.911133
116196,36795,8220.799805,28574.200195
116186,37545,8973.268555,28571.731445
116191,37170,8600.409180,28569.590820
116189,37320,8756.271484,28563.728516
116192,37095,8540.406250,28554.593750


## Xem do quan trong cua cac dac trung

Cell nay lay `feature_importances_` tu `XGBoost` de xem model dang dua nhieu vao dac trung nao nhat. Day khong phai la chan ly tuyet doi, nhung no la mot diem bat dau tot de quyet dinh nen giu hay bo bot feature.

In [21]:
feature_importance_df = pd.DataFrame({
    "feature": feature_columns,
    "importance": xgb_model.feature_importances_,
}).sort_values("importance", ascending=False)

display(feature_importance_df.reset_index(drop=True))

,feature,importance
0,observed_points,0.154752
1,distance_to_destination,0.148101
2,distance_from_start,0.094365
3,progress_ratio,0.087068
4,destination_latitude,0.068500
5,recent_speed_avg_kmh,0.060670
6,avg_speed_since_start_kmh,0.055479
7,recent_speed_kmh,0.050432
8,start_hour,0.049811
9,start_longitude,0.048265


## Thu nghiem ablation voi top feature

Cell nay train lai `XGBoost` voi tap feature top-k theo importance de xem bo dac trung gon hon co giu duoc chat luong hay khong. Neu mot bo nho hon cho ket qua tuong duong hoac tot hon, no la dau hieu rang mot so feature hien tai khong can thiet.

In [22]:
top_k_values = [5, 8, 10, len(feature_columns)]
ablation_rows = []

for top_k in top_k_values:
    selected_features = feature_importance_df.head(top_k)["feature"].tolist()

    ablation_model = XGBRegressor(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1,
    )

    ablation_model.fit(X_train[selected_features], y_train)
    ablation_predictions = ablation_model.predict(X_valid[selected_features])

    ablation_mae = mean_absolute_error(y_valid, ablation_predictions)
    ablation_rmse = mean_squared_error(y_valid, ablation_predictions) ** 0.5

    ablation_rows.append({
        "top_k": top_k,
        "features": ", ".join(selected_features),
        "mae_seconds": ablation_mae,
        "rmse_seconds": ablation_rmse,
        "mae_minutes": ablation_mae / 60,
        "rmse_minutes": ablation_rmse / 60,
    })

ablation_df = pd.DataFrame(ablation_rows)
display(ablation_df)

,top_k,features,mae_seconds,rmse_seconds,mae_minutes,rmse_minutes
0,5,"observed_points, distance_to_destination, dist...",387.401398,1440.477308,6.456690,24.007955
1,8,"observed_points, distance_to_destination, dist...",372.771088,1381.605000,6.212851,23.026750
2,10,"observed_points, distance_to_destination, dist...",380.229919,1457.745863,6.337165,24.295764
3,16,"observed_points, distance_to_destination, dist...",375.631195,1441.624648,6.260520,24.027077


## Chot bo 6 dac trung tot nhat

Cell nay chi tao them mot bien rieng cho bo 6 feature tot nhat theo ket qua importance va ablation hien tai. Flow cu van duoc giu nguyen.

In [23]:
top_6_features = [
    "observed_points",
    "distance_to_destination",
    "distance_from_start",
    "progress_ratio",
    "recent_speed_avg_kmh",
    "avg_speed_since_start_kmh",
]

display(pd.DataFrame({"top_6_features": top_6_features}))

,top_6_features
0,observed_points
1,distance_to_destination
2,distance_from_start
3,progress_ratio
4,recent_speed_avg_kmh
5,avg_speed_since_start_kmh


## Train `XGBoost` tren `top_6_features`

Cell nay train lai `XGBoost` chi voi bo 6 dac trung tot nhat da chot o tren. Muc tieu la so sanh truc tiep voi mo hinh dang dung toan bo feature hien tai.

In [24]:
xgb_top6_model = XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)

xgb_top6_model.fit(X_train[top_6_features], y_train)
xgb_top6_valid_predictions = xgb_top6_model.predict(X_valid[top_6_features])

xgb_top6_mae = mean_absolute_error(y_valid, xgb_top6_valid_predictions)
xgb_top6_rmse = mean_squared_error(y_valid, xgb_top6_valid_predictions) ** 0.5

print(f"XGBoost Top-6 Validation MAE: {xgb_top6_mae:.2f} seconds ({xgb_top6_mae / 60:.2f} minutes)")
print(f"XGBoost Top-6 Validation RMSE: {xgb_top6_rmse:.2f} seconds ({xgb_top6_rmse / 60:.2f} minutes)")

xgb_top6_comparison_df = X_valid[top_6_features].copy()
xgb_top6_comparison_df["actual_eta_seconds"] = y_valid.values
xgb_top6_comparison_df["predicted_eta_seconds"] = xgb_top6_valid_predictions
xgb_top6_comparison_df["absolute_error_seconds"] = (
    xgb_top6_comparison_df["actual_eta_seconds"] - xgb_top6_comparison_df["predicted_eta_seconds"]
).abs()

display(xgb_top6_comparison_df.head(10))

XGBoost Top-6 Validation MAE: 374.22 seconds (6.24 minutes)
XGBoost Top-6 Validation RMSE: 1335.66 seconds (22.26 minutes)


,observed_points,distance_to_destination,distance_from_start,progress_ratio,recent_speed_avg_kmh,avg_speed_since_start_kmh,actual_eta_seconds,predicted_eta_seconds,absolute_error_seconds
176,3,1.366707,0.109239,0.074013,25.243586,13.108728,225,407.108582,182.108582
177,8,0.612460,0.808925,0.569110,38.698648,27.734568,150,262.941833,112.941833
178,13,0.164638,1.335275,0.890235,20.677150,26.705507,75,118.607491,43.607491
200,3,2.801136,0.216420,0.071720,28.403099,25.970437,495,588.261414,93.261414
201,8,2.011678,1.118778,0.357385,54.734207,38.358119,420,442.362518,22.362518
202,13,1.502861,1.712238,0.532562,36.000089,34.244759,345,375.005829,30.005829
203,18,1.198841,2.218786,0.649218,34.102044,31.324042,270,301.215393,31.215393
204,23,0.922032,2.587889,0.737307,34.903556,28.231519,195,240.966125,45.966125
205,28,0.337478,3.075268,0.901113,40.940495,27.335713,120,144.565308,24.565308
206,33,0.083356,2.913663,0.972187,30.638532,21.852475,45,78.275208,33.275208


## Train `XGBoost log-target` tren `top_6_features`

Cell nay giu nguyen y tuong `log1p(target)`, nhung chi dung bo 6 dac trung tot nhat. Day la phep so sanh can bang nhat de xem bo feature rut gon co hop voi huong log-target hay khong.

In [25]:
xgb_log_top6_model = XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)

xgb_log_top6_model.fit(X_train[top_6_features], y_train_log)
xgb_log_top6_valid_predictions = np.expm1(xgb_log_top6_model.predict(X_valid[top_6_features]))
xgb_log_top6_valid_predictions = np.clip(xgb_log_top6_valid_predictions, a_min=0, a_max=None)

xgb_log_top6_mae = mean_absolute_error(y_valid, xgb_log_top6_valid_predictions)
xgb_log_top6_rmse = mean_squared_error(y_valid, xgb_log_top6_valid_predictions) ** 0.5

print(f"XGBoost Log-Target Top-6 Validation MAE: {xgb_log_top6_mae:.2f} seconds ({xgb_log_top6_mae / 60:.2f} minutes)")
print(f"XGBoost Log-Target Top-6 Validation RMSE: {xgb_log_top6_rmse:.2f} seconds ({xgb_log_top6_rmse / 60:.2f} minutes)")

xgb_log_top6_comparison_df = X_valid[top_6_features].copy()
xgb_log_top6_comparison_df["actual_eta_seconds"] = y_valid.values
xgb_log_top6_comparison_df["predicted_eta_seconds"] = xgb_log_top6_valid_predictions
xgb_log_top6_comparison_df["absolute_error_seconds"] = (
    xgb_log_top6_comparison_df["actual_eta_seconds"] - xgb_log_top6_comparison_df["predicted_eta_seconds"]
).abs()

display(xgb_log_top6_comparison_df.head(10))

XGBoost Log-Target Top-6 Validation MAE: 331.88 seconds (5.53 minutes)
XGBoost Log-Target Top-6 Validation RMSE: 1344.44 seconds (22.41 minutes)


,observed_points,distance_to_destination,distance_from_start,progress_ratio,recent_speed_avg_kmh,avg_speed_since_start_kmh,actual_eta_seconds,predicted_eta_seconds,absolute_error_seconds
176,3,1.366707,0.109239,0.074013,25.243586,13.108728,225,341.458557,116.458557
177,8,0.612460,0.808925,0.569110,38.698648,27.734568,150,178.162888,28.162888
178,13,0.164638,1.335275,0.890235,20.677150,26.705507,75,77.304626,2.304626
200,3,2.801136,0.216420,0.071720,28.403099,25.970437,495,509.275146,14.275146
201,8,2.011678,1.118778,0.357385,54.734207,38.358119,420,361.897095,58.102905
202,13,1.502861,1.712238,0.532562,36.000089,34.244759,345,318.237823,26.762177
203,18,1.198841,2.218786,0.649218,34.102044,31.324042,270,255.098663,14.901337
204,23,0.922032,2.587889,0.737307,34.903556,28.231519,195,205.234100,10.234100
205,28,0.337478,3.075268,0.901113,40.940495,27.335713,120,111.869797,8.130203
206,33,0.083356,2.913663,0.972187,30.638532,21.852475,45,50.256878,5.256878


## Thu nghiem tuning cho `XGBoost log-target`

Cell nay thu mot vai cau hinh co chu dich de xem model chung co the tot hon ma khong can doi feature set hay khong.


In [26]:
xgb_tuning_configs = [
    {
        "config_name": "baseline_log_top6",
        "n_estimators": 500,
        "max_depth": 6,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_weight": 1,
        "reg_alpha": 0.0,
        "reg_lambda": 1.0,
    },
    {
        "config_name": "deeper_more_regularized",
        "n_estimators": 700,
        "max_depth": 8,
        "learning_rate": 0.03,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_weight": 3,
        "reg_alpha": 0.1,
        "reg_lambda": 2.0,
    },
    {
        "config_name": "shallower_stable",
        "n_estimators": 900,
        "max_depth": 4,
        "learning_rate": 0.03,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "min_child_weight": 5,
        "reg_alpha": 0.0,
        "reg_lambda": 2.0,
    },
    {
        "config_name": "balanced_regularized",
        "n_estimators": 800,
        "max_depth": 5,
        "learning_rate": 0.03,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "min_child_weight": 3,
        "reg_alpha": 0.1,
        "reg_lambda": 3.0,
    },
]

xgb_tuning_rows = []
best_tuned_model = None
best_tuned_predictions = None
best_tuned_config_name = None
best_tuned_mae = float("inf")

for config in xgb_tuning_configs:
    model = XGBRegressor(
        n_estimators=config["n_estimators"],
        max_depth=config["max_depth"],
        learning_rate=config["learning_rate"],
        subsample=config["subsample"],
        colsample_bytree=config["colsample_bytree"],
        min_child_weight=config["min_child_weight"],
        reg_alpha=config["reg_alpha"],
        reg_lambda=config["reg_lambda"],
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1,
    )

    model.fit(X_train[top_6_features], y_train_log)
    valid_predictions = np.expm1(model.predict(X_valid[top_6_features]))
    valid_predictions = np.clip(valid_predictions, a_min=0, a_max=None)

    mae = mean_absolute_error(y_valid, valid_predictions)
    rmse = mean_squared_error(y_valid, valid_predictions) ** 0.5

    xgb_tuning_rows.append({
        "config_name": config["config_name"],
        "mae_seconds": mae,
        "rmse_seconds": rmse,
        "mae_minutes": mae / 60,
        "rmse_minutes": rmse / 60,
        "n_estimators": config["n_estimators"],
        "max_depth": config["max_depth"],
        "learning_rate": config["learning_rate"],
        "subsample": config["subsample"],
        "colsample_bytree": config["colsample_bytree"],
        "min_child_weight": config["min_child_weight"],
        "reg_alpha": config["reg_alpha"],
        "reg_lambda": config["reg_lambda"],
    })

    if mae < best_tuned_mae:
        best_tuned_mae = mae
        best_tuned_model = model
        best_tuned_predictions = valid_predictions
        best_tuned_config_name = config["config_name"]

xgb_tuning_df = pd.DataFrame(xgb_tuning_rows).sort_values("mae_seconds").reset_index(drop=True)
print(f"Best tuned config: {best_tuned_config_name}")
display(xgb_tuning_df)


Best tuned config: baseline_log_top6


,config_name,mae_seconds,rmse_seconds,mae_minutes,rmse_minutes,n_estimators,max_depth,learning_rate,subsample,colsample_bytree,min_child_weight,reg_alpha,reg_lambda
0,baseline_log_top6,331.875519,1344.439149,5.531259,22.407319,500,6,0.05,0.80,0.80,1,0.0,1.0
1,shallower_stable,332.051819,1338.416088,5.534197,22.306935,900,4,0.03,0.90,0.90,5,0.0,2.0
2,balanced_regularized,332.928864,1345.166625,5.548814,22.419444,800,5,0.03,0.85,0.85,3,0.1,3.0
3,deeper_more_regularized,334.443115,1347.292145,5.574052,22.454869,700,8,0.03,0.80,0.80,3,0.1,2.0


## Xem ket qua cua cau hinh tuned tot nhat

Cell nay giu lai du doan cua cau hinh co `MAE` tot nhat de ban so sanh voi baseline `XGBoost log-target top-6`.


In [27]:
best_tuned_comparison_df = X_valid[top_6_features].copy()
best_tuned_comparison_df["actual_eta_seconds"] = y_valid.values
best_tuned_comparison_df["predicted_eta_seconds"] = best_tuned_predictions
best_tuned_comparison_df["absolute_error_seconds"] = (
    best_tuned_comparison_df["actual_eta_seconds"] - best_tuned_comparison_df["predicted_eta_seconds"]
).abs()

baseline_log_top6_mae = mean_absolute_error(y_valid, xgb_log_top6_valid_predictions)
baseline_log_top6_rmse = mean_squared_error(y_valid, xgb_log_top6_valid_predictions) ** 0.5
best_tuned_rmse = mean_squared_error(y_valid, best_tuned_predictions) ** 0.5

xgb_tuning_summary_df = pd.DataFrame([
    {
        "model": "baseline_log_top6",
        "mae_seconds": baseline_log_top6_mae,
        "rmse_seconds": baseline_log_top6_rmse,
        "mae_minutes": baseline_log_top6_mae / 60,
        "rmse_minutes": baseline_log_top6_rmse / 60,
    },
    {
        "model": best_tuned_config_name,
        "mae_seconds": best_tuned_mae,
        "rmse_seconds": best_tuned_rmse,
        "mae_minutes": best_tuned_mae / 60,
        "rmse_minutes": best_tuned_rmse / 60,
    },
])

display(xgb_tuning_summary_df)
display(best_tuned_comparison_df.head(10))


,model,mae_seconds,rmse_seconds,mae_minutes,rmse_minutes
0,baseline_log_top6,331.875519,1344.439149,5.531259,22.407319
1,baseline_log_top6,331.875519,1344.439149,5.531259,22.407319


,observed_points,distance_to_destination,distance_from_start,progress_ratio,recent_speed_avg_kmh,avg_speed_since_start_kmh,actual_eta_seconds,predicted_eta_seconds,absolute_error_seconds
176,3,1.366707,0.109239,0.074013,25.243586,13.108728,225,341.458557,116.458557
177,8,0.612460,0.808925,0.569110,38.698648,27.734568,150,178.162888,28.162888
178,13,0.164638,1.335275,0.890235,20.677150,26.705507,75,77.304626,2.304626
200,3,2.801136,0.216420,0.071720,28.403099,25.970437,495,509.275146,14.275146
201,8,2.011678,1.118778,0.357385,54.734207,38.358119,420,361.897095,58.102905
202,13,1.502861,1.712238,0.532562,36.000089,34.244759,345,318.237823,26.762177
203,18,1.198841,2.218786,0.649218,34.102044,31.324042,270,255.098663,14.901337
204,23,0.922032,2.587889,0.737307,34.903556,28.231519,195,205.234100,10.234100
205,28,0.337478,3.075268,0.901113,40.940495,27.335713,120,111.869797,8.130203
206,33,0.083356,2.913663,0.972187,30.638532,21.852475,45,50.256878,5.256878
